# 02 · Concept Drift & Performance Monitoring — *Is the model still right?*

**MLOps Monitoring · Coaching Session · Module 3 (Machine Learning & GenAI)**

---

In notebook 01 the **inputs** mostly looked normal: same flat sizes, same towns, same flat types. Only remaining lease drifted, because flats age.

Now we do the real test. **Real prices have come in. We compare them with what the model predicted.**

> **Analogy — the chicken rice stall 🍗**
> In 2019 a plate of chicken rice cost S$3.50. Today the *same plate* (same rice, same chicken, same stall) costs S$5.
> The **input did not change**. The **link between the input and the price** did.
> That is **concept drift**. You **cannot** see it by looking at the plates (inputs) alone — you need the receipts (real prices).

| | Data drift (notebook 01) | Concept drift (this notebook) |
|---|---|---|
| What changed? | The **inputs** — what kind of flats are sold | The **rule** linking inputs → price |
| HDB example | Flats sold today have shorter remaining leases | Same 4-room Tampines flat now costs much more |
| Need real prices to see it? | ❌ No | ✅ Yes |

## 0 · Setup — load data and the live model

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mlops_utils import (load_hdb, make_model, score, register_model, set_champion,
                         load_version, NUM, CAT, FEATURES, TARGET, DEPLOY_DATE)

pd.set_option("display.float_format", "{:,.1f}".format)

df = load_hdb()
reference  = df[df.date <  DEPLOY_DATE]
production = df[df.date >= DEPLOY_DATE].copy()

version = "v1"                                       # the model that went live in Jan 2022
try:
    model = load_version(version)                    # saved in notebook 01
except Exception:                                    # notebook 01 not run yet -> train it here
    model = make_model().fit(reference[FEATURES], reference[TARGET])
    register_model(model, "v1", notes="first release", trained_until="2021-12")
    set_champion("v1")
print(f"Checking model {version}. Production deals to check: {len(production):,}")

---
## Part A · The monthly scorecard 📋

Each month, real prices arrive (HDB resale prices are published by **registration date**, a few weeks after the deal). We compare them with the model's predictions using three numbers:

| Metric | Plain English | Analogy |
|---|---|---|
| **MAE** (S$) | On average, how many dollars off? | A weather app that is "on average 2°C off" |
| **MAPE** (%) | On average, how many **percent** off? Fair across S$300k and S$1.2M flats | "Off by 10%" means the same for a cheap or an expensive flat |
| **Bias** (S$) | Average of *signed* errors. **−** = predicts too low, **+** = too high | A bathroom scale that always shows **3 kg less** than you really weigh |

**MAE says how big the mistakes are. Bias says whether they always lean the same way.** A model can have small MAE and big bias, or the other way round.

In [ ]:
production["pred"]  = model.predict(production[FEATURES])
production["error"] = production["pred"] - production[TARGET]          # + over-predict, - under-predict
production["abs_pct_error"] = production["error"].abs() / production[TARGET] * 100

# 🔧 YOUR TURN: bias is the plain average of the signed error
monthly = production.groupby("month").agg(
    MAE=("error", lambda e: e.abs().mean()),
    MAPE=("abs_pct_error", "mean"),
    bias=("error", ____),
    deals=("error", "size"),
)
monthly.head(6)

<details><summary>▸ Hint</summary>

Use the pandas aggregation name for the average: `"mean"`.
</details>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
monthly["MAPE"].plot(ax=axes[0], marker=".", color="tab:red")
axes[0].set_title("MAPE (%) — how far off, on average")
axes[0].set_ylabel("%")
(monthly["bias"] / 1000).plot(ax=axes[1], marker=".", color="tab:blue")
axes[1].axhline(0, color="black", lw=1)
axes[1].set_title("Bias (S$ thousands) — below 0 = predicts too LOW")
for ax in axes:
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

first3 = monthly.head(3)["MAPE"].mean()
last3  = monthly.tail(3)["MAPE"].mean()
print(f"MAPE in the first 3 months after launch: {first3:.1f}%")
print(f"MAPE in the latest 3 months:             {last3:.1f}%")

**What do you notice?**
* MAPE starts around 10% and climbs to almost **30%**.
* Bias is **negative and growing**: the model predicts **too low, every single month**. It is a scale that is stuck in 2021.
* Around 2025 the climb **flattens out**, because the market itself slowed down (prices rose less).

---
## Part B · Inputs look normal, error explodes → concept drift 🔍

Let's put the **input drift** (from notebook 01) and the **error** on the same chart.

In [ ]:
def psi(expected, actual, bins=10):
    edges = np.unique(np.quantile(expected, np.linspace(0, 1, bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    e = np.clip(np.histogram(expected, edges)[0] / len(expected), 1e-4, None)
    a = np.clip(np.histogram(actual, edges)[0] / len(actual), 1e-4, None)
    return np.sum((a - e) * np.log(a / e))

quarters = production.date.dt.to_period("Q")
by_q = pd.DataFrame({
    "PSI floor area": production.groupby(quarters)["floor_area_sqm"].apply(lambda s: psi(reference["floor_area_sqm"], s)),
    "PSI storey":     production.groupby(quarters)["storey"].apply(lambda s: psi(reference["storey"], s)),
    "MAPE %":         production.groupby(quarters)["abs_pct_error"].mean(),
})

fig, ax1 = plt.subplots(figsize=(12, 4))
by_q[["PSI floor area", "PSI storey"]].plot(ax=ax1, marker="o")
ax1.axhline(0.1, ls="--", color="grey")
ax1.set_ylim(0, 0.5)
ax1.set_ylabel("PSI (inputs)")
ax2 = ax1.twinx()
by_q["MAPE %"].plot(ax=ax2, color="tab:red", marker="s", lw=3, label="MAPE % (right axis)")
ax2.set_ylabel("MAPE %")
ax2.legend(loc="upper center")
plt.title("Inputs stay calm (PSI under the 0.1 line) while the error keeps climbing")
plt.show()

**This is the key picture of the session.** A team that only monitors *input* drift would see 🟢 green dashboards for five years while the model gets worse every month.

### The same flat, different price
Let's follow one *type* of flat over time: **4-room, Tampines, 90–95 sqm**. Same kind of flat, same town, same size.

In [ ]:
# 🔧 YOUR TURN (optional): try a different town / flat type you know
TOWN, FLAT_TYPE, SIZE = "TAMPINES", "4 ROOM", (90, 95)

same_flat = df[(df.town == TOWN) & (df.flat_type == FLAT_TYPE) & df.floor_area_sqm.between(*SIZE)]
yearly = same_flat.groupby(same_flat.date.dt.year)[TARGET].median()

# What does our frozen model predict for these same flats, year by year?
model_says = pd.Series(model.predict(same_flat[FEATURES]), index=same_flat.index)
model_yearly = model_says.groupby(same_flat.date.dt.year).median()

ax = (yearly / 1000).plot(marker="o", figsize=(10, 4), label="Real median price")
(model_yearly / 1000).plot(ax=ax, color="tab:red", ls="--", marker="x", label=f"Model {version} prediction (median)")
ax.axvline(2021.5, color="grey", ls=":")
ax.set_ylabel("S$ thousands")
ax.set_title(f"{FLAT_TYPE}, {TOWN}, {SIZE[0]}–{SIZE[1]} sqm")
ax.legend()
plt.show()
change = (yearly.iloc[-1] / yearly.loc[2021] - 1) * 100
print(f"Median 2021: S${yearly.loc[2021]:,.0f} -> {yearly.index[-1]}: S${yearly.iloc[-1]:,.0f}  ({change:+.0f}%)")

**Same kind of flat, much higher price.** Before launch (left of the dotted line) the model follows the real prices. After launch, its answer **stays flat** — it learned the 2017–2021 *rule* and keeps using it.

### Why did the rule change? Real events
| When | Event | Effect on the price rule |
|---|---|---|
| 2020–2022 | COVID delays BTO completions; families who can't wait buy resale | More demand → higher prices for the *same* flat |
| Dec 2021 | Cooling measures (tighter loan limits) | Tries to slow prices — a *policy* change the model knows nothing about |
| Sep 2022 | Private home owners must wait 15 months before buying resale HDB; HDB loan limit cut to 80% | Changes *who* can buy and *how much* they can borrow |
| Aug 2024 | HDB loan limit cut from 80% to 75% | Buyers need more cash up front |

**Kinds of concept drift**

| Type | Analogy | HDB example |
|---|---|---|
| **Sudden** | A road closes overnight | A new cooling measure takes effect on a fixed date |
| **Gradual** | Hair growing — you only notice after months | Prices creeping up month after month (what we see here) |
| **Seasonal / recurring** | Durian season, Chinese New Year | Fewer deals around festive periods |

---
## Part C · Break it down — averages hide problems 🔎

One overall MAPE is like a school's average grade: it can hide a class that is failing. Look at the **latest 6 months**, split by flat type and by town.

We use **bias %** = average of (error ÷ real price) × 100, so a S$1M flat and a S$400k flat count fairly.

In [ ]:
latest = production[production.date >= production.date.max() - pd.DateOffset(months=5)].copy()
latest["bias_pct"] = latest["error"] / latest[TARGET] * 100

def segment_report(data, by):
    return (data.groupby(by)
                .agg(deals=("error", "size"), MAPE=("abs_pct_error", "mean"),
                     bias_pct=("bias_pct", "mean"), bias_S=("error", "mean"))
                .sort_values("bias_pct"))

print(f"Overall bias, latest 6 months: {latest.bias_pct.mean():.1f}%\n")
segment_report(latest, "flat_type")

In [ ]:
# 🔧 YOUR TURN: flag segments that are badly off.
#   Rule: at least 30 deals AND |bias %| above the threshold you choose
THRESHOLD = ____

towns = segment_report(latest, "town")
flagged = towns[(towns.deals >= 30) & (towns.bias_pct.abs() > THRESHOLD)]
print(f"{len(flagged)} of {len(towns)} towns are off by more than {THRESHOLD}% on average:")
flagged

<details><summary>▸ Hint</summary>

Any sensible number works — try `30`, then `25`. A lower threshold flags more towns. In real life, agree the threshold with the people who *use* the predictions (e.g. the agents).
</details>

In [ ]:
# Heatmap: bias % by town x flat type (latest 6 months)
pivot = latest.pivot_table(index="town", columns="flat_type", values="bias_pct", aggfunc="mean")
counts = latest.pivot_table(index="town", columns="flat_type", values="bias_pct", aggfunc="size")
pivot = pivot.where(counts >= 10)          # hide cells with too few deals to trust
pivot = pivot[[c for c in ["2 ROOM", "3 ROOM", "4 ROOM", "5 ROOM", "EXECUTIVE"] if c in pivot]]

fig, ax = plt.subplots(figsize=(8, 10))
im = ax.imshow(pivot.values, cmap="RdBu", vmin=-50, vmax=50, aspect="auto")
ax.set_xticks(range(len(pivot.columns)), pivot.columns)
ax.set_yticks(range(len(pivot.index)), pivot.index)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        v = pivot.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=8)
plt.colorbar(im, label="bias % (red = predicts too LOW)")
plt.title("Where is model v1 most wrong? (latest 6 months)")
plt.tight_layout()
plt.show()

**What do you notice?**
* Every cell is red: the model is too low **everywhere**.
* **Surprise:** in *percent*, the worst towns are not the expensive central ones. They are towns like **Woodlands, Punggol and Hougang**, where prices rose fastest since 2021.
* In *dollars*, the biggest misses are **bigger flats and Central Area**: S$200k–300k per flat.

That is why we look at **both** % and S$, and at **segments**, not just one average.

---
## Part D · The "ceiling" problem of tree models 📏

Our model (gradient boosted trees, like Random Forest and XGBoost) predicts by **averaging prices it has already seen**. So it **can never predict higher than the highest prices in its training data**.

> **Analogy — the measuring tape 📐** If your tape only goes up to 1.36 m, you can't measure a 1.8 m person. You just write "1.36 m" and move on.

In [ ]:
train_max = reference[TARGET].max()
pred_max  = production["pred"].max()
million_real = (production[TARGET] >= 1_000_000).sum()
million_pred = (production["pred"] >= 1_000_000).sum()

print(f"Highest price in training data (2017–2021): S${train_max:,.0f}")
print(f"Highest price the model EVER predicted:     S${pred_max:,.0f}")
print(f"Highest real price since launch:            S${production[TARGET].max():,.0f}\n")
print(f"Million-dollar deals since launch:  real = {million_real:,}   predicted = {million_pred:,}")

sample = production[production.date.dt.year == 2025].sample(4000, random_state=0)
plt.figure(figsize=(6, 6))
plt.scatter(sample[TARGET] / 1e3, sample["pred"] / 1e3, s=4, alpha=0.3)
lim = [150, 1700]
plt.plot(lim, lim, "k--", label="perfect prediction")
plt.axhline(pred_max / 1e3, color="tab:red", label="model's ceiling")
plt.xlabel("Real price (S$k)"); plt.ylabel("Predicted (S$k)"); plt.legend()
plt.title("2025: predictions flatten out at the top")
plt.show()

**Fixes (pick one, test it in notebook 03):**
1. **Retrain often**, so the "tape" keeps getting longer.
2. **Predict a ratio instead of a price** — e.g. price ÷ that month's median price — then multiply back. The model learns *"this flat is 1.4× the market"*, which stays true even when the market moves.
3. Use a model that can extrapolate (e.g. linear regression with a time trend) for part of the job.

---
## Part E · Always too low? The "slow clock" check 🕰️

One bad month is normal. **Being wrong in the same direction month after month** is a pattern.

> **Analogy — the slow clock ⏰** A clock that is 1 minute fast one day and 1 minute slow the next is fine. A clock that is **slow every single day** is broken, even if it is "only" a few minutes slow.

**The sign check:** look at the last 12 months. Count months where the model was **too high** (+1) and **too low** (−1), and add them up.

* A healthy model is too high about half the time and too low about half the time → the score stays near **0**.
* A model that always leans one way → the score heads to **+12** or **−12**.
* **Alarm when the score reaches ±8** (i.e. 10 or more of the last 12 months in the same direction). By pure luck this happens only about 4% of the time, so an alarm means something real.

*(Forecasting textbooks use a similar tool called the **tracking signal** = sum of errors ÷ average error size. Same idea, but noisier with few months.)*

In [ ]:
def lean_score(errors, window=12):
    errors = pd.Series(errors)
    # 🔧 YOUR TURN: np.sign gives +1 for "too high", -1 for "too low". Add them up over the window.
    return np.sign(errors).rolling(window, min_periods=1).____()

rng = np.random.default_rng(1)
healthy = pd.Series(rng.normal(0, 30_000, len(monthly)), index=monthly.index)   # random ups and downs

lean = pd.DataFrame({"healthy model (errors bounce)": lean_score(healthy).values,
                     f"model {version} (real)": lean_score(monthly["bias"]).values},
                    index=monthly.index)
ax = lean.plot(figsize=(12, 4), marker=".")
ax.axhline(8, color="grey", ls="--"); ax.axhline(-8, color="grey", ls="--"); ax.set_ylim(-13, 13)
ax.set_title("Sign check: last 12 months, (+1 too high) + (-1 too low). Outside ±8 = consistent lean")
plt.xticks(rotation=45)
plt.show()

first_alarm = lean.index[lean.iloc[:, 1].abs() >= 8][0]
print(f"Model {version} hits the alarm line in: {first_alarm}")

<details><summary>▸ Hint</summary>

To add numbers over a rolling window: `.rolling(window).sum()`.
</details>

**What do you notice?**
* The healthy model wanders around 0. It touches the line **once**, by pure luck, then comes straight back. That is the ~4% false alarm. **Act when the score stays at the line**, not on a single touch.
* Model v1 is too low **every single month**. It hits the alarm line after 8 months and stays at −12.
* With the sign check, you know there is a problem in **2022** — years before anyone would have noticed from an overall MAPE report.

💡 Run the same check **per segment** (per flat type, per town) to find *where* the model leans.

---
## Part F · For the time-series side of your project ⏱️

If your group also **forecasts** a series (e.g. median monthly resale price), monitoring works the same way: wait for the real value, compare, track the bias.

Below, a forecast **made in Dec 2021** (straight-line trend from 2019–2021) is compared with what really happened.

In [ ]:
median_price = df.groupby("date")[TARGET].median()
history = median_price[(median_price.index >= "2019-01-01") & (median_price.index < DEPLOY_DATE)]
future  = median_price[median_price.index >= DEPLOY_DATE]

# Forecast made on 31 Dec 2021: straight-line trend through 2019–2021
t_hist = np.arange(len(history))
slope, intercept = np.polyfit(t_hist, history.values, 1)
t_future = np.arange(len(history), len(history) + len(future))
frozen_forecast = pd.Series(intercept + slope * t_future, index=future.index)

# Rolling forecast: each month, simply forecast "same as last month" (re-made monthly)
rolling_forecast = median_price.shift(1).loc[future.index]

ax = (median_price / 1e3).plot(figsize=(12, 4), label="Actual median", color="black")
(frozen_forecast / 1e3).plot(ax=ax, label="Forecast frozen in Dec 2021", ls="--")
(rolling_forecast / 1e3).plot(ax=ax, label="Re-forecast every month (naive)", alpha=0.7)
ax.axvline(pd.Timestamp(DEPLOY_DATE), color="grey", ls=":")
ax.set_ylabel("S$ thousands"); ax.legend()
plt.show()

signals = {}
for name, fc in [("Frozen forecast", frozen_forecast), ("Re-forecast monthly", rolling_forecast)]:
    err = fc - future
    signals[name] = lean_score(err)
    print(f"{name:<22} MAPE = {(err.abs() / future).mean() * 100:5.1f}%")

ax = pd.DataFrame(signals).plot(figsize=(12, 3.5))
ax.axhline(8, color="grey", ls="--"); ax.axhline(-8, color="grey", ls="--"); ax.set_ylim(-13, 13)
ax.set_title("Sign check of the two forecasts (outside ±8 = consistent lean)")
plt.show()

**What do you notice?**
* The **frozen** forecast is too low in 2022–2023, then **too high** from late 2023 as its straight line rises faster than the market. Its sign check sits at the alarm lines for long stretches.
* The **re-forecast-every-month** version has a much smaller error, because it always starts from the latest real value.
* But look at its sign check in **2023–2024**: it leans **low**. While prices rose, "same as last month" was **always one step behind**. Each error was small, but the *pattern* was clear. When prices flattened in 2025, it came back towards 0. The fix: add a trend to the forecast.

For time series, *re-forecasting with the newest data* is the same idea as *retraining* — the topic of notebook 03.

---
## Part G · 🔧 YOUR TURN — scorecard for *your group's* model

In [ ]:
# my_test = ...   # newer months your model did NOT train on, with real prices
# my_test["pred"]  = my_model.predict(my_test[my_features])
# my_test["error"] = my_test["pred"] - my_test["resale_price"]
# my_test["abs_pct_error"] = my_test["error"].abs() / my_test["resale_price"] * 100
# my_test["bias_pct"] = my_test["error"] / my_test["resale_price"] * 100
#
# print(my_test.groupby("month").agg(MAPE=("abs_pct_error", "mean"), bias=("error", "mean")))
# print(segment_report(my_test, "flat_type"))
# print(lean_score(my_test.groupby("month")["error"].mean()).tail())

---
## ✍️ Reflect (2 minutes, with your group)
1. Your input dashboards were 🟢 green for years. What would have caught this problem earlier?
2. Which segment would you **warn users about** today, before any fix is ready?
3. Is your group's model a tree model? Where is its "ceiling"?

### Key takeaways
* **Concept drift** = same inputs, different answer. You need **real outcomes** (prices) to see it.
* Track **MAPE (how far)** *and* **bias (which way)** every month.
* **Break it down** by segment — the average hides the worst cases.
* **Sign check beyond ±8** (10+ of the last 12 months in one direction) = consistently over/under-predicting → act.
* **Tree models can't predict above what they've seen.** In a rising market they under-predict at the top.